In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog qiskit-serverless
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# 最初のQiskit Serverlessプログラムを作成する

<Accordion>
<AccordionItem title="パッケージバージョン">

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降を使用することをお勧めします。

```
qiskit[all]~=1.3.1
qiskit-ibm-runtime~=0.34.0
qiskit-aer~=0.15.1
qiskit-serverless~=0.18.1
qiskit-ibm-catalog~=0.2
qiskit-addon-sqd~=0.8.1
qiskit-addon-utils~=0.1.0
qiskit-addon-mpf~=0.2.0
qiskit-addon-aqc-tensor~=0.1.2
qiskit-addon-obp~=0.1.0
scipy~=1.15.0
pyscf~=2.8.0
```
</AccordionItem>
</Accordion>
> **Tip:** **Qiskit Serverlessはアップグレード中であり、その機能は急速に変化しています。** この開発フェーズでは、[Qiskit Serverless GitHub](https://qiskit.github.io/qiskit-serverless/index.html)ページでリリースノートと最新ドキュメントをご確認ください。

この例では、`qiskit-serverless`ツールを使用して並列トランスパイルプログラムを作成し、`qiskit-ibm-catalog`を実装してプログラムをIBM Quantum Platformに再利用可能なリモートサービスとしてアップロードする方法を示します。

## ワークフローの概要
1. ローカルディレクトリと空のプログラムファイル（`./source_files/transpile_remote.py`）を作成する
1. アップロード時にQiskit Serverlessで回路をトランスパイルするコードをプログラムに追加する
1. `qiskit-ibm-catalog`を使用してQiskit Serverlessに認証する
1. プログラムをQiskit Serverlessにアップロードする

プログラムをアップロードしたら、[Run your first Qiskit Serverless workload remotely](/guides/serverless-run-first-workload)ガイドに従って回路をトランスパイルするために実行できます。

## 例: Qiskit Serverlessによるリモートトランスパイル
この例では、Qiskit Serverlessにアップロードした際に、指定された`backend`とターゲット`optimization_level`に対して`circuit`をトランスパイルするプログラムファイルの作成と追記手順を説明します。

:::tip

Qiskit Serverlessでは、ワークロードの`.py`ファイルを専用のディレクトリに設定する必要があります。以下の構成は良い慣行の例です:

In [ ]:
# This cell is hidden from users, it creates a new folder
from pathlib import Path

Path("./source_files").mkdir(exist_ok=True)

In [ ]:
%%writefile ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this
# locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit.transpiler import generate_preset_pass_manager

def transpile_remote(circuit, optimization_level, backend):
    """Transpiles an abstract circuit into an ISA circuit for a given backend."""
    pass_manager = generate_preset_pass_manager(
        optimization_level=optimization_level,
		backend=backend
    )
    isa_circuit = pass_manager.run(circuit)
    return isa_circuit

### Add code to get program arguments

Now add the following code to your program file, which sets up program arguments.

Your initial `transpile_remote.py` has three inputs: `circuits`, `backend_name`, and `optimization_level`. Serverless is currently limited to only accept serializable inputs and outputs. For this reason, you cannot pass in `backend` directly, so use `backend_name` as a string instead.

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this
# locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit_serverless import get_arguments, save_result, distribute_task, get

# Get program arguments
arguments = get_arguments()
circuits = arguments.get("circuits")
backend_name = arguments.get("backend_name")
optimization_level = arguments.get("optimization_level")

### プログラム引数を取得するコードを追加する
次に、プログラム引数を設定する以下のコードをプログラムファイルに追加します。

初期の`transpile_remote.py`には3つの入力があります: `circuits`、`backend_name`、`optimization_level`です。Serverlessは現在、シリアライズ可能な入力と出力のみを受け付けるという制限があります。そのため、`backend`を直接渡すことはできないため、代わりに文字列として`backend_name`を使用します。

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this
# locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend(backend_name)

### バックエンドを呼び出すコードを追加する
`QiskitRuntimeService`を使用してバックエンドを呼び出す以下のコードをプログラムファイルに追加します。

以下のコードは、`QiskitRuntimeService.save_account`を使用して認証情報を保存する手順を既に実行済みであることを前提とし、特に指定しない限りデフォルトで保存済みのアカウントを読み込みます。詳細については、[ログイン認証情報を保存する](/guides/save-credentials)および[Qiskit Runtimeサービスアカウントを初期化する](/guides/initialize-account)を参照してください。

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this
# locally in a notebook), running this cell saves to disk rather than executing the code.

# Each circuit is being transpiled and will populate the array
results = [
    transpile_remote(circuit, 1, backend)
    for circuit in circuits
]

save_result({
    "transpiled_circuits": results
})

### トランスパイルするコードを追加する
最後に、以下のコードをプログラムファイルに追加します。このコードは渡されたすべての`circuits`に対して`transpile_remote()`を実行し、`transpiled_circuits`を結果として返します:

In [13]:
from qiskit_ibm_catalog import QiskitServerless, QiskitFunction

# Authenticate to the remote cluster and submit the pattern for remote execution
serverless = QiskitServerless()

<span id="upload-to-qiskit-serverless"></span>
### Qiskit Serverlessに認証する
`qiskit-ibm-catalog`を使用して、APIキーで`QiskitServerless`に認証します（`QiskitRuntimeService`のAPIキーを使用するか、[IBM Quantum Platformダッシュボード](https://quantum.cloud.ibm.com)で新しいAPIキーを作成できます）。

In [8]:
transpile_remote_demo = QiskitFunction(
    title="transpile_remote_serverless",
    entrypoint="transpile_remote.py",
    working_dir="./source_files/",
)

In [9]:
serverless.upload(transpile_remote_demo)

QiskitFunction(transpile_remote_serverless)

### Verify upload

To check if it successfully uploaded, use `serverless.list()`, as in the following code:

In [10]:
# Get program from serverless.list() that matches the title of the one we uploaded
next(
    program
    for program in serverless.list()
    if program.title == "transpile_remote_serverless"
)

QiskitFunction(transpile_remote_serverless)

In [11]:
# This cell is hidden from users, it checks the program uploaded correctly
assert _.title == "transpile_remote_serverless"  # noqa: F821

In [12]:
# This cell is hidden from users, it checks the program executes correctly
from time import sleep
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
transpile_remote_serverless = serverless.load("transpile_remote_serverless")
job = transpile_remote_serverless.run(
    circuits=[qc],
    backend="ibm_sherbrooke",
    optimization_level=1,
)
while True:
    sleep(5)
    status = job.status()
    if status not in ["QUEUED", "INITIALIZING", "RUNNING", "DONE"]:
        raise Exception(
            f"Unexpected job status: '{status}'\n"
            + "Here are the logs:\n"
            + job.logs()
        )
    if status == "DONE":
        break

## Next steps

<Admonition type="info" title="Recommendations">

- Learn how to pass inputs and run your program remotely in the [Run your first Qiskit Serverless workload remotely](/docs/guides/serverless-run-first-workload) topic.

</Admonition>